# 2.1b Data Collection — Citra Sentinel-2A Kecamatan Kamal (versi perbaikan)

Mengunduh **Sentinel-2 L2A** untuk **Kecamatan Kamal** lewat openEO (Copernicus Data Space).

**Input:** `kamal.geojson` di folder yang sama (atau upload otomatis di Colab).
**Output:** `s2_kamal.tif` + `s2_kamal_metadata.json`.

Perbaikan dari versi sebelumnya:
- `temporal_extent` memakai **jendela tanggal** (akhir bersifat *exclusive*), bukan satu hari.
- Semua fitur GeoJSON digabung (bukan hanya `features[0]`) dan CRS dicek.
- Dimensi waktu di-**median composite** agar GTiff valid dan awan tersisa berkurang.
- Job dibuat eksplisit supaya **log error terlihat** kalau gagal.

In [1]:
%pip install -q openeo folium shapely rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.9 MB/s eta 0:00:00


In [2]:
import os, sys, json
from datetime import date, timedelta

import openeo
import folium
import shapely.geometry
from shapely.ops import unary_union
from shapely.geometry import mapping

## 1. Parameter

In [4]:
GEOJSON_PATH = "kamal.geojson"

TANGGAL_CITRA = "2026-08-15"   # tanggal target (dekat waktu digitasi QGIS)
HARI_SEBELUM = 10              # jendela pencarian: target - N hari
HARI_SESUDAH = 10              # jendela pencarian: target + N hari
MAX_CLOUD_COVER = 20           # % awan maksimum per scene (longgarkan bila data kosong)

BANDS_10M = ["B02", "B03", "B04", "B08"]   # biru, hijau, merah, NIR
BANDS_20M = ["B11", "B12"]                 # SWIR (di-resample ke 10 m)
OUTPUTFILE = "s2_kamal.tif"

## 2. Muat batas wilayah (AOI)

In [6]:
if not os.path.exists(GEOJSON_PATH):
    if "google.colab" in sys.modules:
        from google.colab import files
        print("File 'kamal.geojson' tidak ditemukan. Silakan upload:")
        uploaded = files.upload()
        name = list(uploaded.keys())[0]
        if name != GEOJSON_PATH:
            os.rename(name, GEOJSON_PATH)
    else:
        raise FileNotFoundError(f"'{GEOJSON_PATH}' tidak ditemukan di {os.getcwd()}")

with open(GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

# Gabungkan SEMUA fitur (mis. bila file berisi per-desa)
if geojson_data.get("type") == "FeatureCollection":
    geoms = [shapely.geometry.shape(ft["geometry"]) for ft in geojson_data["features"]]
elif geojson_data.get("type") == "Feature":
    geoms = [shapely.geometry.shape(geojson_data["geometry"])]
else:
    geoms = [shapely.geometry.shape(geojson_data)]

shp = unary_union(geoms)
geometry = mapping(shp)
west, south, east, north = shp.bounds

# Cek CRS: harus derajat (EPSG:4326)
if not (-180 <= west <= 180 and -90 <= south <= 90 and -180 <= east <= 180 and -90 <= north <= 90):
    raise ValueError(
        f"Koordinat tampak bukan derajat (bounds={shp.bounds}). "
        "Reproyeksi GeoJSON ke EPSG:4326 di QGIS lalu ekspor ulang."
    )

print(f"Jumlah fitur : {len(geoms)}")
print(f"Tipe geometri: {shp.geom_type}")
print(f"BBox         : W={west:.5f} S={south:.5f} E={east:.5f} N={north:.5f}")

c = shp.centroid
m = folium.Map(location=[c.y, c.x], zoom_start=12)
folium.GeoJson(geojson_data, name="Kecamatan Kamal").add_to(m)
m

Jumlah fitur : 1
Tipe geometri: Polygon
BBox         : W=112.69127 S=-7.17571 E=112.77036 N=-7.09748


## 3. Koneksi openEO
Akan muncul link/kode login Copernicus; selesaikan login di browser.

In [7]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=EJPU-JXOL 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


## 4. Definisikan cube

In [8]:
d = date.fromisoformat(TANGGAL_CITRA)
temporal_extent = [
    (d - timedelta(days=HARI_SEBELUM)).isoformat(),
    (d + timedelta(days=HARI_SESUDAH + 1)).isoformat(),   # akhir exclusive -> +1 hari
]
spatial_extent = {"west": west, "south": south, "east": east, "north": north, "crs": "EPSG:4326"}
print("Temporal extent:", temporal_extent)

cube = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=temporal_extent,
    spatial_extent=spatial_extent,
    bands=BANDS_10M + BANDS_20M,
    max_cloud_cover=MAX_CLOUD_COVER,
)

# Potong mengikuti poligon Kamal
cube = cube.filter_spatial(geometries=geometry)

# Hilangkan dimensi waktu (median composite) -> GTiff 2D + band
cube = cube.reduce_dimension(dimension="t", reducer="median")
print("Cube siap:", BANDS_10M + BANDS_20M)

Temporal extent: ['2026-08-05', '2026-08-26']
Cube siap: ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']


## 5. Jalankan job & unduh
Proses bisa beberapa menit. Jika gagal, log error dicetak otomatis.

In [9]:
job = cube.create_job(title="Sentinel-2A Kecamatan Kamal", out_format="GTiff")
try:
    job.start_and_wait()
    job.get_results().download_file(OUTPUTFILE)
    print(f"Selesai: {OUTPUTFILE}")
except Exception as e:
    print("JOB GAGAL:", repr(e))
    print("Job ID:", job.job_id)
    for entry in job.logs():
        if entry.get("level") in ("error", "warning"):
            print(entry.get("level", "").upper(), "-", entry.get("message"))
    raise

0:00:00 Job 'j-2609290843204d78b240b22b4d690463': send 'start'
0:00:06 Job 'j-2609290843204d78b240b22b4d690463': queued (progress 0%)
0:00:12 Job 'j-2609290843204d78b240b22b4d690463': queued (progress 0%)
0:00:18 Job 'j-2609290843204d78b240b22b4d690463': queued (progress 0%)
0:00:27 Job 'j-2609290843204d78b240b22b4d690463': queued (progress 0%)
0:00:37 Job 'j-2609290843204d78b240b22b4d690463': queued (progress 0%)
0:00:51 Job 'j-2609290843204d78b240b22b4d690463': running (progress N/A)
0:01:07 Job 'j-2609290843204d78b240b22b4d690463': running (progress N/A)
0:01:27 Job 'j-2609290843204d78b240b22b4d690463': running (progress N/A)
0:01:51 Job 'j-2609290843204d78b240b22b4d690463': running (progress N/A)
0:02:22 Job 'j-2609290843204d78b240b22b4d690463': running (progress N/A)
0:03:01 Job 'j-2609290843204d78b240b22b4d690463': finished (progress 100%)
Selesai: s2_kamal.tif


## 6. Verifikasi & metadata

In [10]:
import rasterio

assert os.path.exists(OUTPUTFILE), f"{OUTPUTFILE} tidak terunduh!"
print(f"Ukuran file: {os.path.getsize(OUTPUTFILE) / 1e6:.1f} MB")

with rasterio.open(OUTPUTFILE) as src:
    print("CRS        :", src.crs)
    print("Ukuran     :", src.width, "x", src.height, "piksel")
    print("Jumlah band:", src.count)
    print("Resolusi   :", src.res)
    print("Bounds     :", src.bounds)
    band_desc = list(src.descriptions)
    print("Deskripsi band:", band_desc)
    arr = src.read(1)
    print("Band 1 min/max:", arr.min(), arr.max())

metadata = {
    "koleksi": "SENTINEL2_L2A",
    "tanggal_target": TANGGAL_CITRA,
    "temporal_extent": temporal_extent,
    "komposit": "median",
    "bands": BANDS_10M + BANDS_20M,
    "max_cloud_cover_pct": MAX_CLOUD_COVER,
    "bbox": spatial_extent,
    "aoi_file": GEOJSON_PATH,
    "output": OUTPUTFILE,
    "job_id": job.job_id,
}
with open("s2_kamal_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata disimpan ke s2_kamal_metadata.json")

Ukuran file: 3.4 MB
CRS        : EPSG:32749
Ukuran     : 875 x 865 piksel
Jumlah band: 6
Resolusi   : (10.0, 10.0)
Bounds     : BoundingBox(left=686760.0, bottom=9206470.0, right=695510.0, top=9215120.0)
Deskripsi band: ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
Band 1 min/max: -32768 8096
Metadata disimpan ke s2_kamal_metadata.json


## Unduh Berkas Data

*   **s2_kamal.tif** — Komposit Sentinel-2 L2A Kecamatan Kamal (3,3 MB):  
    <a href="s2_kamal.tif" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh s2_kamal.tif</a>

*   **s2_kamal_metadata.json** — Metadata citra (tanggal, band, CRS):  
    <a href="s2_kamal_metadata.json" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh s2_kamal_metadata.json</a>

*   **kamal.geojson** — Batas wilayah Kecamatan Kamal (GeoJSON):  
    <a href="kamal.geojson" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal.geojson</a>
